In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# highest level of qualification by sex
# To download original dataset go to - 
# https://fingertips.phe.org.uk/
mortality_rate_csv = r"N:\Geodatabase\Raw_Data\OHID Health\OHID_Mortality_Rate.csv"

## 2. Input Link to MSOA 2011 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::middle-layer-super-output-areas-december-2011-boundaries-ew-bfe-v3/about

In [3]:
msoa2011_shapefile = r"N:\Geodatabase\Raw_Data\Census 2021\Boundaries\Middle_Layer_Super_Output_Areas_Dec_2011_Boundaries_Full_Extent_BFE_EW_V3_2022_46367968394682200\MSOA_2011_EW_BFE_V3.shp"

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [4]:
mortality_rate_raw_data_df = pd.read_csv(mortality_rate_csv)

# STEP 1: FILTER FOR ROWS WHERE 'AREA TYPE' EQUALS 'MSOA' and 'TIME PERIOD' EQUALS '2016 - 20'
mortality_rate_df = mortality_rate_raw_data_df[
    (mortality_rate_raw_data_df['Area Type'] == 'MSOA') &
    (mortality_rate_raw_data_df['Time period'] == '2016 - 20')
]

# STEP 2: KEEP ONLY THE SPECIFIED COLUMNS
columns_to_keep = [
    'Indicator Name', 'Area Code', 'Value', 'Count', 'Denominator'
]

mortality_rate_df = mortality_rate_df[columns_to_keep]

# STEP 3: RENAME THE SPECIFIED COLUMNS
mortality_rate_df.rename(columns={
    'Indicator Name': 'indicator_name',
    'Area Code': 'msoa11cd',    
}, inplace=True)

# STEP 4: CREATE SEPARATE DATAFRAMES FOR EACH UNIQUE VALUE IN 'INDICATOR NAME'
unique_indicators = mortality_rate_df['indicator_name'].unique()
dataframes = {}

for indicator in unique_indicators:
    dataframes[indicator] = mortality_rate_df[mortality_rate_df['indicator_name'] == indicator]

#STEP 5: CREATE INDIVIDUAL DATAFRAME FROM DICTIONARY
deaths_from_all_cancer_all_ages_df = dataframes['Deaths from all cancer, all ages, standardised mortality ratio']
deaths_from_all_cancer_under_75_years_df = dataframes['Deaths from all cancer, under 75 years, standardised mortality ratio']
deaths_from_all_causes_all_ages_df = dataframes['Deaths from all causes, all ages, standardised mortality ratio']
deaths_from_all_causes_all_ages_under_75_years_df = dataframes['Deaths from all causes, under 75 years, standardised mortality ratio']
deaths_from_causes_considered_preventable_under_75_years_df = dataframes['Deaths from causes considered preventable, under 75 years, standardised mortality ratio']
deaths_from_circulatory_disease_all_ages_df = dataframes['Deaths from circulatory disease, all ages, standardised mortality ratio']
deaths_from_circulatory_disease_all_ages_under_75_years_df = dataframes['Deaths from circulatory disease, under 75 years, standardised mortality ratio']
deaths_from_coronary_heart_disease_all_ages_df = dataframes['Deaths from coronary heart disease, all ages, standardised mortality ratio']
deaths_from_respiratory_diseases_all_ages_df = dataframes['Deaths from respiratory diseases, all ages, standardised mortality ratio']
deaths_from_stroke_df = dataframes['Deaths from stroke, all ages, standardised mortality ratio']

# STEP 6: RENAME VALUE, COUNT AND DENOMINATOR FIELDS
# Define renaming mappings for each DataFrame
rename_mappings = {
    "deaths_from_all_cancer_all_ages_df": {
        'Value': 'deaths_from_all_cancer_all_ages_perc',
        'Count': 'deaths_from_all_cancer_all_ages_count',
        'Denominator': 'deaths_from_all_cancer_all_ages_denominator',
    },
    "deaths_from_all_cancer_under_75_years_df": {
        'Value': 'deaths_from_all_cancer_under_75_years_perc',
        'Count': 'deaths_from_all_cancer_under_75_years_count',
        'Denominator': 'deaths_from_all_cancer_under_75_years_denominator',
    },
    "deaths_from_all_causes_all_ages_df": {
        'Value': 'deaths_from_all_causes_all_ages_perc',
        'Count': 'deaths_from_all_causes_all_ages_count',
        'Denominator': 'deaths_from_all_causes_all_ages_denominator',
    },
    "deaths_from_all_causes_all_ages_under_75_years_df": {
        'Value': 'deaths_from_all_causes_all_ages_under_75_years_perc',
        'Count': 'deaths_from_all_causes_all_ages_under_75_years_count',
        'Denominator': 'deaths_from_all_causes_all_ages_under_75_years_denominator',
    },
    "deaths_from_causes_considered_preventable_under_75_years_df": {
        'Value': 'deaths_from_causes_considered_preventable_under_75_years_perc',
        'Count': 'deaths_from_causes_considered_preventable_under_75_years_count',
        'Denominator': 'deaths_from_causes_considered_preventable_under_75_years_denominator',
    },
    "deaths_from_circulatory_disease_all_ages_df": {
        'Value': 'deaths_from_circulatory_disease_all_ages_perc',
        'Count': 'deaths_from_circulatory_disease_all_ages_count',
        'Denominator': 'deaths_from_circulatory_disease_all_ages_denominator',
    },
    "deaths_from_circulatory_disease_all_ages_under_75_years_df": {
        'Value': 'deaths_from_circulatory_disease_all_ages_under_75_years_perc',
        'Count': 'deaths_from_circulatory_disease_all_ages_under_75_years_count',
        'Denominator': 'deaths_from_circulatory_disease_all_ages_under_75_years_denominator',
    },
    "deaths_from_coronary_heart_disease_all_ages_df": {
        'Value': 'deaths_from_coronary_heart_disease_all_ages_perc',
        'Count': 'deaths_from_coronary_heart_disease_all_ages_count',
        'Denominator': 'deaths_from_coronary_heart_disease_all_ages_denominator',
    },
    "deaths_from_respiratory_diseases_all_ages_df": {
        'Value': 'deaths_from_respiratory_diseases_all_ages_perc',
        'Count': 'deaths_from_respiratory_diseases_all_ages_count',
        'Denominator': 'deaths_from_respiratory_diseases_all_ages_denominator',
    },
    "deaths_from_stroke_df": {
        'Value': 'deaths_from_stroke_perc',
        'Count': 'deaths_from_stroke_count',
        'Denominator': 'deaths_from_stroke_denominator',
    }
}

# Iterate over DataFrames and apply renaming
for df_name, renames in rename_mappings.items():
    locals()[df_name].rename(columns=renames, inplace=True)

# STEP 7: MERGE DATAFRAMES
# drop fields to avoid duplication
# list of columns to drop
columns_to_drop = [
    'indicator_name', 
]

# Drop the columns from the dataframe
# Drop columns from each dataframe in Step 5
deaths_from_all_cancer_all_ages_df = deaths_from_all_cancer_all_ages_df.drop(columns=['indicator_name'], errors='ignore')
deaths_from_all_cancer_under_75_years_df = deaths_from_all_cancer_under_75_years_df.drop(columns=columns_to_drop, errors='ignore')
deaths_from_all_causes_all_ages_df = deaths_from_all_causes_all_ages_df.drop(columns=columns_to_drop, errors='ignore')
deaths_from_all_causes_all_ages_under_75_years_df = deaths_from_all_causes_all_ages_under_75_years_df.drop(columns=columns_to_drop, errors='ignore')
deaths_from_causes_considered_preventable_under_75_years_df = deaths_from_causes_considered_preventable_under_75_years_df.drop(columns=columns_to_drop, errors='ignore')
deaths_from_circulatory_disease_all_ages_df = deaths_from_circulatory_disease_all_ages_df.drop(columns=columns_to_drop, errors='ignore')
deaths_from_circulatory_disease_all_ages_under_75_years_df = deaths_from_circulatory_disease_all_ages_under_75_years_df.drop(columns=columns_to_drop, errors='ignore')
deaths_from_coronary_heart_disease_all_ages_df = deaths_from_coronary_heart_disease_all_ages_df.drop(columns=columns_to_drop, errors='ignore')
deaths_from_respiratory_diseases_all_ages_df = deaths_from_respiratory_diseases_all_ages_df.drop(columns=columns_to_drop, errors='ignore')
deaths_from_stroke_df = deaths_from_stroke_df.drop(columns=columns_to_drop, errors='ignore')

# Merge the dataframes in the specified order

# First merge: deaths_from_all_causes_all_ages_df and deaths_from_all_causes_all_ages_under_75_years_df
step1_mortality_rate_df = pd.merge(
    deaths_from_all_causes_all_ages_df,
    deaths_from_all_causes_all_ages_under_75_years_df,
    how='left',
    on='msoa11cd'
)

# Second merge: step1_mortality_rate_df with deaths_from_causes_considered_preventable_under_75_years_df
step2_mortality_rate_df = pd.merge(
    step1_mortality_rate_df,
    deaths_from_causes_considered_preventable_under_75_years_df,
    how='left',
    on='msoa11cd'
)

# Third merge: step2_mortality_rate_df with deaths_from_all_cancer_all_ages_df
step3_mortality_rate_df = pd.merge(
    step2_mortality_rate_df,
    deaths_from_all_cancer_all_ages_df,
    how='left',
    on='msoa11cd'
)

# Fourth merge: step3_mortality_rate_df with deaths_from_all_cancer_under_75_years_df
step4_mortality_rate_df = pd.merge(
    step3_mortality_rate_df,
    deaths_from_all_cancer_under_75_years_df,
    how='left',
    on='msoa11cd'
)

# Fifth merge: step4_mortality_rate_df with deaths_from_circulatory_disease_all_ages_df
step5_mortality_rate_df = pd.merge(
    step4_mortality_rate_df,
    deaths_from_circulatory_disease_all_ages_df,
    how='left',
    on='msoa11cd'
)

# Sixth merge: step5_mortality_rate_df with deaths_from_circulatory_disease_all_ages_under_75_years_df
step6_mortality_rate_df = pd.merge(
    step5_mortality_rate_df,
    deaths_from_circulatory_disease_all_ages_under_75_years_df,
    how='left',
    on='msoa11cd'
)

# Seventh merge: step6_mortality_rate_df with deaths_from_coronary_heart_disease_all_ages_df
step7_mortality_rate_df = pd.merge(
    step6_mortality_rate_df,
    deaths_from_coronary_heart_disease_all_ages_df,
    how='left',
    on='msoa11cd'
)

# Eighth merge: step7_mortality_rate_df with deaths_from_respiratory_diseases_all_ages_df
step8_mortality_rate_df = pd.merge(
    step7_mortality_rate_df,
    deaths_from_respiratory_diseases_all_ages_df,
    how='left',
    on='msoa11cd'
)

# Ninth merge: step8_mortality_rate_df with deaths_from_stroke_df
final_mortality_rate_df = pd.merge(
    step8_mortality_rate_df,
    deaths_from_stroke_df,
    how='left',
    on='msoa11cd'
)

# Display the updated dataframe
final_mortality_rate_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19180\1248836311.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  mortality_rate_raw_data_df = pd.read_csv(mortality_rate_csv)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19180\1248836311.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  locals()[df_name].rename(columns=renames, inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19180\1248836311.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  locals()[df_name].rename(columns=renames, inplace=True)
C:\Users\abhimanya.ach

,msoa11cd,deaths_from_all_causes_all_ages_perc,deaths_from_all_causes_all_ages_count,deaths_from_all_causes_all_ages_denominator,deaths_from_all_causes_all_ages_under_75_years_perc,deaths_from_all_causes_all_ages_under_75_years_count,deaths_from_all_causes_all_ages_under_75_years_denominator,deaths_from_causes_considered_preventable_under_75_years_perc,deaths_from_causes_considered_preventable_under_75_years_count,deaths_from_causes_considered_preventable_under_75_years_denominator,deaths_from_all_cancer_all_ages_perc,deaths_from_all_cancer_all_ages_count,deaths_from_all_cancer_all_ages_denominator,deaths_from_all_cancer_under_75_years_perc,deaths_from_all_cancer_under_75_years_count,deaths_from_all_cancer_under_75_years_denominator,deaths_from_circulatory_disease_all_ages_perc,deaths_from_circulatory_disease_all_ages_count,deaths_from_circulatory_disease_all_ages_denominator,deaths_from_circulatory_disease_all_ages_under_75_years_perc,deaths_from_circulatory_disease_all_ages_under_75_years_count,deaths_from_circulatory_disease_all_ages_under_75_years_denominator,deaths_from_coronary_heart_disease_all_ages_perc,deaths_from_coronary_heart_disease_all_ages_count,deaths_from_coronary_heart_disease_all_ages_denominator,deaths_from_respiratory_diseases_all_ages_perc,deaths_from_respiratory_diseases_all_ages_count,deaths_from_respiratory_diseases_all_ages_denominator,deaths_from_stroke_perc,deaths_from_stroke_count,deaths_from_stroke_denominator
0,E02000001,50.247355,202.0,402.011213,61.290593,80.0,130.525739,71.472594,42.0,58.763783,65.183666,69.0,105.854740,67.478951,34.0,50.386082,55.775547,53.85,96.547687,64.748007,18.0,27.800083,73.347225,30.0,40.901343,36.508637,18.916,51.812397,35.621603,7.952,22.323532
1,E02000002,99.344041,343.0,345.264794,132.848007,105.0,79.037693,120.340653,42.5,35.316411,112.130843,89.0,79.371561,135.386944,39.0,28.806323,92.922057,77.83,83.758370,142.866305,23.0,16.098967,83.598469,28.0,33.493436,102.475182,46.804,45.673498,73.317476,14.792,20.175272
2,E02000003,94.424566,350.0,370.666250,109.105081,137.0,125.567021,98.458641,55.5,56.368846,89.812507,87.0,96.868469,90.736002,42.0,46.288132,104.476530,91.85,87.914482,143.473019,37.0,25.788821,96.669062,36.0,37.240457,115.420217,53.860,46.664268,87.573001,17.755,20.274514
3,E02000004,113.410620,389.0,343.001386,114.107227,93.0,81.502287,136.803642,50.0,36.548735,112.164249,90.0,80.239471,116.986639,36.0,30.772745,95.596398,79.80,83.475949,87.915240,15.0,17.061888,107.077043,36.0,33.620652,129.338926,58.888,45.529990,93.337770,18.670,20.002621
4,E02000005,98.243799,265.0,269.737126,109.845725,111.0,101.050815,130.078418,59.0,45.357255,108.499515,78.0,71.889722,110.294040,40.0,36.266692,101.255854,63.86,63.067959,162.164843,33.0,20.349664,110.991609,30.0,27.029070,129.804013,42.860,33.019010,68.158421,9.787,14.359194


## 4. Load GIS shapefile 

In [5]:
# Attempt to read from the server first, fallback to local path if it fails
msoa2011boundary_df = gpd.read_file(msoa2011_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
msoa2011boundary_df.head()

Shapefile loaded successfully from the server.


,MSOA11CD,MSOA11NM,BNG_E,BNG_N,LONG_,LAT,Shape_Leng,GlobalID,geometry
0,E02000001,City of London 001,532384,181355,-0.093490,51.5156,9651.221144,ae98212b-e51e-43cb-8eb8-9a9edc228ea7,"POLYGON ((532153.703 182165.155, 532158.250 18..."
1,E02000002,Barking and Dagenham 001,548267,189685,0.138756,51.5865,8306.888230,6cc3f460-0089-4d6f-85ef-c05f4e78d547,"POLYGON ((548881.304 190819.980, 548881.125 19..."
2,E02000003,Barking and Dagenham 002,548259,188520,0.138149,51.5760,9359.512342,38e8a480-1669-4ca6-a07b-b628b1c7c302,"POLYGON ((548958.555 189072.176, 548954.517 18..."
3,E02000004,Barking and Dagenham 003,551004,186412,0.176828,51.5564,8475.840309,232da89a-870b-44f8-a2e5-d7dc5ad28f0f,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E02000005,Barking and Dagenham 004,548733,186824,0.144267,51.5607,7321.657104,3926d87b-f86e-44c9-88a1-0c49e220a56f,"POLYGON ((549237.051 187627.941, 549241.319 18..."


## 5. GIS data management

### Remove Rename fields

In [6]:
#Drop and rename fields
msoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)
msoa2011boundary_df.rename(columns = {'MSOA11CD':'msoa11cd','MSOA11NM':'msoa11nm'}, inplace = True)
msoa2011boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19180\931186528.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  msoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)


,msoa11cd,msoa11nm,geometry
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18..."
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19..."
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18..."
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [7]:
#MSOA11 LOOKUP
msoa11_lookup_df = pd.read_excel(open(r"N:\Geodatabase\Raw_Data\Look up tables\MSOA_(2011)_to_MSOA_(2021)_to_Local_Authority_District_(2022)_Lookup_for_England_and_Wales_7815823766405070629(3).xlsx", 'rb'), sheet_name='MSOA_(2011)_to_MSOA_(2021)__0') 

columns_to_keep = [
    'MSOA11CD','LAD22CD','LAD22NM'
]

#drop unwanted fields
msoa11_lookup_cleaned_df = msoa11_lookup_df[columns_to_keep]

#rename msoa11cd to make merging cleaner
msoa11_lookup_cleaned_df.rename(columns={
    'MSOA11CD': 'msoa11cd',
    'LAD22CD': 'lad22cd',
    'LAD22NM': 'lad22nm',
}, inplace=True)

msoa11_lookup_cleaned_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19180\2762448710.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msoa11_lookup_cleaned_df.rename(columns={


,msoa11cd,lad22cd,lad22nm
0,E02000001,E09000001,City of London
1,E02000002,E09000002,Barking and Dagenham
2,E02000003,E09000002,Barking and Dagenham
3,E02000004,E09000002,Barking and Dagenham
4,E02000005,E09000002,Barking and Dagenham


In [8]:
msoa2011boundary_df = pd.merge(msoa2011boundary_df, msoa11_lookup_cleaned_df, how = 'left', on = 'msoa11cd')
msoa2011boundary_df.head()

,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham


#### LAD22 to REGION22

In [9]:
# Define the file path for lad to region
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19180\2710224882.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [10]:
msoa2011boundary_df = pd.merge(msoa2011boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
msoa2011boundary_df.head()

,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm,rgn22cd,rgn22nm
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London,E12000007,London
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham,E12000007,London
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham,E12000007,London
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham,E12000007,London
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham,E12000007,London


### Add data management fields

In [11]:
# Add new data management fields with specified values
msoa2011boundary_df['data_source'] = "Office for Health Improvement & Disparities"
msoa2011boundary_df['data_resolution'] = "MSOA 2011"
msoa2011boundary_df['data_time_period'] = "2016 - 20"
msoa2011boundary_df['data_web_link'] = "https://fingertips.phe.org.uk/"
msoa2011boundary_df.head()

,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/


### Update Area

In [12]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in msoa2011boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    msoa2011boundary_df['area_ha'] = msoa2011boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in msoa2011boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
msoa2011boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,315.147533
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,216.156066
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,214.151524
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,249.294610
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,118.795417


# 7. Combine Geometry and data

In [14]:
ohid_mortality_rate_msoa2011_gdb_df = pd.merge(msoa2011boundary_df, final_mortality_rate_df, how = 'left', on='msoa11cd')
ohid_mortality_rate_msoa2011_gdb_df.head()

,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,deaths_from_all_causes_all_ages_perc,deaths_from_all_causes_all_ages_count,deaths_from_all_causes_all_ages_denominator,deaths_from_all_causes_all_ages_under_75_years_perc,deaths_from_all_causes_all_ages_under_75_years_count,deaths_from_all_causes_all_ages_under_75_years_denominator,deaths_from_causes_considered_preventable_under_75_years_perc,deaths_from_causes_considered_preventable_under_75_years_count,deaths_from_causes_considered_preventable_under_75_years_denominator,deaths_from_all_cancer_all_ages_perc,deaths_from_all_cancer_all_ages_count,deaths_from_all_cancer_all_ages_denominator,deaths_from_all_cancer_under_75_years_perc,deaths_from_all_cancer_under_75_years_count,deaths_from_all_cancer_under_75_years_denominator,deaths_from_circulatory_disease_all_ages_perc,deaths_from_circulatory_disease_all_ages_count,deaths_from_circulatory_disease_all_ages_denominator,deaths_from_circulatory_disease_all_ages_under_75_years_perc,deaths_from_circulatory_disease_all_ages_under_75_years_count,deaths_from_circulatory_disease_all_ages_under_75_years_denominator,deaths_from_coronary_heart_disease_all_ages_perc,deaths_from_coronary_heart_disease_all_ages_count,deaths_from_coronary_heart_disease_all_ages_denominator,deaths_from_respiratory_diseases_all_ages_perc,deaths_from_respiratory_diseases_all_ages_count,deaths_from_respiratory_diseases_all_ages_denominator,deaths_from_stroke_perc,deaths_from_stroke_count,deaths_from_stroke_denominator
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,315.147533,50.247355,202.0,402.011213,61.290593,80.0,130.525739,71.472594,42.0,58.763783,65.183666,69.0,105.854740,67.478951,34.0,50.386082,55.775547,53.85,96.547687,64.748007,18.0,27.800083,73.347225,30.0,40.901343,36.508637,18.916,51.812397,35.621603,7.952,22.323532
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,216.156066,99.344041,343.0,345.264794,132.848007,105.0,79.037693,120.340653,42.5,35.316411,112.130843,89.0,79.371561,135.386944,39.0,28.806323,92.922057,77.83,83.758370,142.866305,23.0,16.098967,83.598469,28.0,33.493436,102.475182,46.804,45.673498,73.317476,14.792,20.175272
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,214.151524,94.424566,350.0,370.666250,109.105081,137.0,125.567021,98.458641,55.5,56.368846,89.812507,87.0,96.868469,90.736002,42.0,46.288132,104.476530,91.85,87.914482,143.473019,37.0,25.788821,96.669062,36.0,37.240457,115.420217,53.860,46.664268,87.573001,17.755,20.274514
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,249.294610,113.410620,389.0,343.001386,114.107227,93.0,81.502287,136.803642,50.0,36.548735,112.164249,90.0,80.239471,116.986639,36.0,30.772745,95.596398,79.80,83.475949,87.915240,15.0,17.061888,107.077043,36.0,33.620652,129.338926,58.888,45.529990,93.337770,18.670,20.002621
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham,E12000007,London,Office for Health Improvement & Disparities,MSOA 2011,2016 - 20,https://fingertips.phe.org.uk/,118.795417,98.243799,265.0,269.737126,109.845725,111.0,101.050815,130.078418,59.0,45.357255,108.499515,78.0,71.889722,110.294040,40.0,36.266692,101.255854,63.86,63.067959,162.164

In [16]:
ohid_mortality_rate_msoa2011_gdb_df = ohid_mortality_rate_msoa2011_gdb_df.drop_duplicates(subset='msoa11cd', keep='first')

# 9. Upload to geodatabase

In [18]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "ohid_msoa2011_mortality_rate"  # Desired table name
primary_key_column = "msoa11cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [19]:
# Ensure the GeoDataFrame has a valid CRS before writing
if ohid_mortality_rate_msoa2011_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    ohid_mortality_rate_msoa2011_gdb_df.set_crs(epsg=27700, inplace=True)

In [21]:
# Publish the GeoDataFrame to PostGIS
ohid_mortality_rate_msoa2011_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.ohid_msoa2011_mortality_rate
Primary key set on column: msoa11cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.ohid_msoa2011_mortality_rate
